# v2 — the precision study

Companion to `finetuning_assetgen.ipynb` (the v1 recall study). Canonical record is
`ABLATION_LOG_V2.md`.

**Objective: raise precision without letting recall fall.** The recall guard is a decision
rule applied at the ablation table (§2 of the log) — it is deliberately NOT a check inside
generation, because recall is measured against the ground truth and feeding that back would
be selecting on the answers.

**Input regime: comment-stripped RTL only.** No spec summary, no parsed ports/signals. So
the whole of v1's Stage A (PDF extraction, chunking, embeddings, FAISS, RAG summaries and
their QC) is absent from this notebook — it produces an input v2 does not use.

RTL parsing IS kept, because `validate_primary` and the scorer need the parsed element
lists. It reads `parsed_tuning18/` from cache and makes no API calls.

**Exception — arm `v2rp`.** That arm restores the parsed lists to the prompt, annotated
with functional roles by the line-4 parser in `prompts_parse_v2`. It is the only version
in this study whose input regime is not RTL-only. Its closed set is *copied* from
`parsed_tuning18/` and asserted identical, so the scorer and `validate_primary` are
unaffected. See §RP of `ABLATION_LOG_V2.md`.


In [20]:
import os
from dotenv import load_dotenv

print("loaded:", load_dotenv(r"API.env"))   

loaded: True


In [21]:
#config
from pathlib import Path
import json, re, time, hashlib, os

RTL_DIR   = Path("RTL_data")
OUT_DIR   = Path("summaries_tuning18"); OUT_DIR.mkdir(exist_ok=True)
# DEFINED HERE, not in the reference-build cell. In v1 GT_DIR existed only inside the
# "build the reference sets" cell, which points at an external checkout of the LAsset
# results repo -- so scoring could not run at all unless that cell had been run first,
# even though ground_truth/*.json were already on disk. Verified: Stage C raised
# NameError: name 'GT_DIR' is not defined. The reference-build cell re-assigns the same
# value, so it stays correct either way.
GT_DIR    = Path("ground_truth")

GEN_MODEL   = "gpt-5-mini"                         # paper: GPT-5 family

# ===================== ABLATION SWITCH =====================
# Picks which (core prompt, ICL examples) pair Stage B uses. The registry lives in the
# Stage B cell -- add an entry there, set VERSION here, re-run Stage B.
#
# v2 IS THE NAME OF THE STUDY, NOT A KEY IN v1's REGISTRY. v1 already has a "v02" entry
# (the AES-swap ICL variant, never run). These are different notebooks with different
# prompt modules, so there is no collision -- but do not carry a key across.
#
#   v2   prompts_v2.ASSET_V2_BASE + icl_examples_01b_noparse       ROOT of this study
#   v2rp prompts_v2.ASSET_V2_RP   + icl_examples_01b_noparse       parsed lists + roles
#   v2p3 prompts_v2.ASSET_V2_P3   + icl_examples_01b_noparse       parsed lists, v3 parse
#
# v2rp IS THE ONLY VERSION THAT NEEDS A CELL RUN BEFORE STAGE B. Its parsed blocks carry
# role annotations produced by the 'RTL Parsing with functional roles' cell below, which
# writes parsed_roles_tuning18/. Run that cell and read its report BEFORE Stage B.
#
# The baseline is the v01c6p1p2 input regime plus two corrections, both in prompts_v2.py:
#   D  closed-set parity -- RULES no longer refer to parsed blocks that are not sent
#   L  de-leak           -- four real NEORV32 identifiers removed as naming examples;
#                           ctrl_i is a ground-truth ANSWER in 10 of the 41 modules
#
# ARM KEYS ARE ALPHANUMERIC SUFFIXES ONLY -- v2e1, v2e2, never v2_e1.
# EV.collect() globs assets_tuning18_{v}_r*, so a key of the form <baseline>_<tag> has its
# repeat directories collected as EXTRA REPEATS OF THE BASELINE, silently corrupting both.
# Verified: glob("assets_tuning18_vBASE_r*") matches "assets_tuning18_vBASE_r8_r0".
#
# A new arm is CURRENT_BASELINE plus ONE change and pairs against it. Add it to BASELINE in
# the ablation cell, or the default there sends it to CURRENT_BASELINE -- either way it
# never falls through to a multi-variable contrast against VERSIONS[0].
CURRENT_BASELINE = "v2"

VERSION = "v2p3c1"

# Repeats of the SAME prompt. n=3 SCREENS an arm; n=5 before promoting one. v1 established
# that 3 is enough for precision (3-of-5 subsampling, 100% sign agreement) and that 5 is
# needed for per-class recall (port sign agrees only 58% at n=3). This study's guard is on
# recall, so promotion needs 5.
REPEATS = 3

def asset_dir(version=None, rep=0, create=False):
    """Path for one (version, repeat). Nothing is ever overwritten.

    create=False by default: merely naming a directory must not bring it into existence.
    An empty assets_tuning18_<v>_r0 sitting on disk shadows a real earlier run during
    evaluation and scores as all-zeros. Only the generation loop passes create=True.
    """
    d = Path(f"assets_tuning18_{version or VERSION}_r{rep}")
    if create:
        d.mkdir(exist_ok=True)
    return d

print(f"VERSION = {VERSION}   REPEATS = {REPEATS}   -> "
      f"assets_tuning18_{VERSION}_r0..r{REPEATS - 1}")


VERSION = v2p3c1   REPEATS = 3   -> assets_tuning18_v2p3c1_r0..r2


In [22]:
from pathlib import Path

def discover_modules(rtl_dir=RTL_DIR):
    """All .vhd files in the RTL dir — the full 18-module tuning set.
    Only the filename stem (= IP module name) is used at this stage;
    RTL content is not read until Task 3 (Modular RTL Parsing)."""
    return [(p.stem, p) for p in sorted(Path(rtl_dir).glob("*.vhd"))]

modules = discover_modules()
assert len(modules) == 18, f"expected 18, found {len(modules)}"
print([s for s, _ in modules])

['neorv32_boot_rom', 'neorv32_bus', 'neorv32_cache', 'neorv32_cpu', 'neorv32_cpu_cp_cfu', 'neorv32_cpu_cp_muldiv', 'neorv32_cpu_pmp', 'neorv32_debug_dtm', 'neorv32_fifo', 'neorv32_hwspinlock', 'neorv32_imem', 'neorv32_package', 'neorv32_spi', 'neorv32_sys', 'neorv32_trng', 'neorv32_twi', 'neorv32_uart', 'neorv32_wdt']


## RTL parsing

Kept for **scoring and validation only** — v2 does not put the parsed lists in the prompt.
`parse_module` returns the cached `parsed_tuning18/<module>.json` when it exists, so this
makes no API calls on a normal run.

The two cells after it are for **`v2rp` only** and do make API calls: they re-annotate the
same closed set with functional roles into `parsed_roles_tuning18/`, then report the
positive-role rate. Skip both for every other version.

In [23]:
# Cell 3 — the client (OpenAI; OPENAI_API_KEY in env)
import os, time, random, json
from openai import OpenAI

class LLMClient:
    def __init__(self, model=GEN_MODEL, effort="medium", max_retries=6):
        self.c = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
        self.model, self.effort, self.r = model, effort, max_retries
        self.usage = []            # per-call token accounting for cost logging

    def _call(self, system, user, json_mode, max_tokens, effort):
        kw = dict(
            model=self.model,
            instructions=system,
            input=user,
            reasoning={"effort": effort or self.effort},
            max_output_tokens=max_tokens,
        )
        if json_mode:
            kw["text"] = {"format": {"type": "json_object"}}

        wait = 2.0
        for a in range(self.r):
            try:
                r = self.c.responses.create(**kw)
                if getattr(r, "status", None) == "incomplete":
                    # ran out of max_output_tokens (reasoning shares the budget):
                    # surface it, return the (possibly truncated) text and let the
                    # caller's escalation path retry with a bigger budget.
                    why = getattr(getattr(r, "incomplete_details", None), "reason", "?")
                    print(f"    [llm warn] incomplete response ({why}) at max_output_tokens={max_tokens}")
                u = r.usage
                self.usage.append({
                    "model": self.model,
                    "in": u.input_tokens,
                    "out": u.output_tokens,
                    "reasoning": u.output_tokens_details.reasoning_tokens,
                    "cached_in": getattr(u.input_tokens_details, "cached_tokens", 0),
                })
                return r.output_text
            except Exception as e:
                if a == self.r - 1:
                    raise
                time.sleep(wait + random.uniform(0, 1)); wait *= 2

    def complete(self, system, user, max_tokens=4096, effort=None):
        """JSON-mode call (Stage A batches, asset generation)."""
        return self._call(system, user, True, max_tokens, effort)

    def complete_text(self, system, user, max_tokens=4096, effort=None):
        """Plain-text call (markdown technical summaries)."""
        return self._call(system, user, False, max_tokens, effort)

    def cost(self, price_in=0.25, price_out=2.00, price_cached=0.025):
        """USD for calls so far, at per-1M-token prices (default gpt-5-mini)."""
        i = sum(u["in"] - u["cached_in"] for u in self.usage)
        c = sum(u["cached_in"] for u in self.usage)
        o = sum(u["out"] for u in self.usage)
        rt = sum(u["reasoning"] for u in self.usage)
        total = (i*price_in + c*price_cached + o*price_out) / 1e6
        return {"calls": len(self.usage), "in": i, "cached_in": c,
                "out": o, "reasoning": rt, "usd": round(total, 4)}

client = LLMClient()

In [24]:
# ===================== Cell — Modular RTL Parsing (Prm_m) =====================
import importlib, prompts, rtl_parse
importlib.reload(prompts); importlib.reload(rtl_parse)
from prompts import PARSE_PORTS_ANNOTATE_SYSTEM, PARSE_SIGNALS_ANNOTATE_SYSTEM
from rtl_parse import build_record_registry, parse_rtl_file

# package + all modules -> so bus_req_t / bus_rsp_t (declared in neorv32_package) are known everywhere
RECORDS = build_record_registry(sorted(RTL_DIR.glob("*.vhd")))
print(f"{len(RECORDS)} record types; bus_req_t={'bus_req_t' in RECORDS}, bus_rsp_t={'bus_rsp_t' in RECORDS}")

PARSE_DIR = Path("parsed_tuning18"); PARSE_DIR.mkdir(exist_ok=True)

def _loads(txt):
    try:
        return json.loads(txt)
    except Exception:
        pass
    m = re.search(r"\{.*\}", txt, re.S)   # salvage a complete object if the tail was cut
    if m:
        try:
            return json.loads(m.group(0))
        except Exception:
            pass
    return {}                              # give up gracefully rather than raise


28 record types; bus_req_t=True, bus_rsp_t=True


In [25]:
# RUN-ALL GUARD. The v1 parse. The annotated arms build their own closed set and never read this,
# and this is the only unguarded cell that can issue API calls.
if VERSION not in ("v2p3", "v2p3c1", "v2rp"):
    def _annot_user(stem, entity, rtl, elems, key):
        return (f"MODULE: {stem}\nENTITY: {entity}\n\n"
                f"AUTHORITATIVE {key.upper()} of entity '{entity}' (ground truth from RTL — annotate "
                f"all, add/drop none; names include record fields written verbatim with a dot, "
                f"e.g. 'ctrl.buf_req', 'host_req_i.stb'):\n{json.dumps(elems, indent=2)}\n\n"
                f"VHDL SOURCE:\n{rtl}\n\n"
                f"Return the annotated {key} as a JSON object using the required schema.")

    def _annotate(system, stem, entity, rtl, elems, key, batch=50, max_tokens=8192):
        """Annotate in batches so large record-expanded port lists (e.g. bus io_switch = 596
        ports) don't overflow the model's output and truncate the JSON. A batch that still
        fails to parse degrades its elements to 'unclear' instead of crashing the module."""
        ann = {}
        for i in range(0, len(elems), batch):
            chunk = elems[i:i + batch]
            got = _loads(client.complete(system, _annot_user(stem, entity, rtl, chunk, key),
                                         max_tokens=max_tokens)).get(key, [])
            for a in got:
                if "name" in a:
                    ann[a["name"]] = a
            if not got:
                print(f"    [annot warn] {stem}/{entity} {key} batch {i//batch}: "
                      f"{len(chunk)} elems left unclear")
        return ann

    def annotate_ports(stem, entity, rtl, ports):
        a = _annotate(PARSE_PORTS_ANNOTATE_SYSTEM, stem, entity, rtl, ports, "ports")
        return {n: v.get("function", "unclear from RTL") for n, v in a.items()}

    def annotate_signals(stem, entity, rtl, signals):
        a = _annotate(PARSE_SIGNALS_ANNOTATE_SYSTEM, stem, entity, rtl, signals, "signals")
        return {n: {"kind": v.get("kind", "signal"),
                    "function": v.get("function", "unclear from RTL")} for n, v in a.items()}

    def parse_module(stem, path, records, overwrite=False):
        outp = PARSE_DIR / f"{stem}.json"
        if outp.exists() and not overwrite:
            return json.loads(outp.read_text(encoding="utf-8"))
        d = parse_rtl_file(path, records=records)
        rtl = Path(path).read_text(encoding="utf-8", errors="ignore")
        ports_all, signals_all = [], []
        for ent in d["entities"]:                       # annotate per entity -> no cross-entity name clashes
            en = ent["entity"]
            pf = annotate_ports(stem, en, rtl, ent["ports"])
            sf = annotate_signals(stem, en, rtl, ent["signals"])
            for p in ent["ports"]:
                ports_all.append({"entity": en, **p, "function": pf.get(p["name"], "unclear from RTL")})
            for s in ent["signals"]:
                signals_all.append({"entity": en, **s,
                                    "kind": sf.get(s["name"], {}).get("kind", "signal"),
                                    "function": sf.get(s["name"], {}).get("function", "unclear from RTL")})
        parsed = {"module": stem, "entities": [e["entity"] for e in d["entities"]],
                  "ports": ports_all, "signals": signals_all}
        outp.write_text(json.dumps(parsed, indent=2), encoding="utf-8")
        return parsed

    parsed_all = {}
    for stem, path in modules:
        try:
            parsed_all[stem] = parse_module(stem, path, RECORDS)
            p = parsed_all[stem]
            print(f"[ok  ] {stem:24s} entities={p['entities']}  {len(p['ports'])}p / {len(p['signals'])}s")
        except Exception as e:
            print(f"[err ] {stem:24s} {e}")
else:
    print(f"skipped: this cell is for the v1 parse regime, VERSION is {VERSION}")


skipped: this cell is for the v1 parse regime, VERSION is v2p3c1


In [28]:
# ============ Cell — Modular RTL Parsing (Prm_m), annotated ============
# ALGORITHM 1 LINE 4. Builds the parsed record FROM SCRATCH for the current version:
# rtl_parse extracts every port and signal mechanically from the RTL, then an LLM adds
# meaning to the elements extraction already found. The model never decides membership;
# the regex never guesses at meaning. verify_against_cache raises if the two disagree by a
# single element, so a coverage failure is a hard error and not a judgement call.
#
# Two annotation regimes exist and they are FULLY ISOLATED — separate prompt modules,
# separate drivers, separate output directories, and the imports below are inside their
# branches so a v2p3 run never loads v2rp's code at all:
#
#   v2rp -> parsed_roles_tuning18/   roles from a 152-label CLOSED taxonomy   [superseded]
#   v2p3 -> parsed_v3_tuning18/      OPEN vocabulary, occurrence-driven       [current]
#
# Both write the same schema: entity, name, dir/kind, type, plus function, functionality,
# role, relationship, evidence.
#
# WHAT v2p3 DOES DIFFERENTLY, and why each matters:
#   comment-stripped source   a comment is where a designer states a conclusion the RTL does
#                             not show; an annotator told to read one reports it as observed
#   ENTITY-scoped source      54 names in neorv32_bus are declared in more than one of its
#                             six entities. Sending the whole file made an annotation for one
#                             entity's element a merge of two same-named elements
#   merged ports+signals      one call per entity covers both lists. A registered deviation
#                             from line 4's two parsers; annotate_all(..., merged=False)
#                             restores them. 61 -> 30 calls, 284k -> 166k input tokens
#
# Cached per module — re-running is free for what is already on disk. overwrite=True forces
# re-annotation. SKIP for any other version: it costs API calls and nothing else reads it.
if VERSION in ("v2p3", "v2p3c1"):
    import importlib, prompts_parse_v3, parse_v3
    importlib.reload(prompts_parse_v3); importlib.reload(parse_v3)
    # RECORDS is REQUIRED. Without it the record-typed ports declared in the package do not
    # expand and the closed set silently differs from the one every other stage uses.
    # PARALLEL. The calls are independent by construction -- each is one entity's batch
    # of a disjoint element set, annotated from that entity's own scoped source -- so
    # concurrency changes throughput and nothing else. Verified: the sequential and
    # parallel paths produce byte-identical output on all 18 modules.
    #
    # Serially this stage is ~100 minutes: 30 calls, and the heavy ones emit ~11.5k
    # output tokens apiece. neorv32_bus alone is 10 of them. Lower PARSE_WORKERS if the
    # API rate-limits; the client already retries with backoff, but retries cost wall
    # clock, so throttling here is cheaper than being throttled there.
    PARSE_WORKERS = 6
    parsed_annot = parse_v3.annotate_all_parallel(client.complete, modules,
                                                 records=RECORDS, workers=PARSE_WORKERS)

elif VERSION == "v2rp":
    import importlib, prompts_parse_v2, parse_roles
    importlib.reload(prompts_parse_v2); importlib.reload(parse_roles)
    parsed_annot = parse_roles.annotate_all(client.complete, modules)

else:
    print(f"{VERSION}: not an annotated-parse version -- skipping (nothing downstream "
          f"needs it, and this cell costs API calls)")

standing-rule audit: no corpus identifier in either prompt
18 module(s) to annotate, 30 call(s), 6 workers

  [ 1/30] neorv32_boot_rom/neorv32_boot_rom batch 0: 21 of 21 annotated
  [ 2/30] neorv32_bus/neorv32_bus_reg batch 0: 36 of 36 annotated
  [ 3/30] neorv32_bus/neorv32_bus_gateway batch 0: 100 of 100 annotated
  [ 4/30] neorv32_bus/neorv32_bus_switch batch 0: 62 of 62 annotated
  [ 5/30] neorv32_bus/neorv32_bus_io_switch batch 0: 128 of 128 annotated
  [ 6/30] neorv32_bus/neorv32_bus_io_switch batch 1: 128 of 128 annotated
  [ 7/30] neorv32_bus/neorv32_bus_io_switch batch 2: 128 of 128 annotated
  [ 8/30] neorv32_bus/neorv32_bus_io_switch batch 3: 128 of 128 annotated
  [ 9/30] neorv32_bus/neorv32_bus_io_switch batch 4: 70 of 70 annotated
  [10/30] neorv32_bus/neorv32_bus_amo_rmw batch 0: 51 of 51 annotated
  [11/30] neorv32_bus/neorv32_bus_amo_rvs batch 0: 39 of 39 annotated
  [12/30] neorv32_cache/neorv32_cache_memory batch 0: 22 of 22 annotated
  [13/30] neorv32_cache/neorv32_

In [29]:
# ============ THE GATE — read this before paying for a generation run ============
# v2p3 (open vocabulary). There is no label to count, so the gate is the share of elements
# whose annotation used VERDICT OR HEDGING vocabulary. The v3 stage is forbidden to write
# about security at all, so this rate measures directly whether it stayed on its side of the
# line. TARGET ~0. Anything else means the parser has begun doing the asset stage's job, and
# the asset stage would be reading its own rubric back out of its input.
#
# Two more numbers on the same report, each worth as much as the headline:
#
#   isolated %      elements with no substantiated coupling. The relationships are what the
#                   asset stage reasons over; a high rate means the occurrence scan under-ran.
#   GATES:SOURCES   GATES is the branch-condition edge that a value-flow reading cannot see.
#                   Measured against the ground truth: 80% of true dotted-signal assets
#                   appear in a condition versus 28% of the non-assets, a 52-point gap. If
#                   GATES is scarce here, the most discriminating instruction in the prompt
#                   is not landing and the asset stage loses that separation.
#
# Any non-zero `no_role`, `no_evidence` or `dead_batch` under "other issues" means elements
# would reach the asset stage unannotated. Re-run those modules with overwrite=True first.
if VERSION in ("v2p3", "v2p3c1"):
    import importlib, parse_v3, parse_audit
    importlib.reload(parse_v3); importlib.reload(parse_audit)
    parse_v3.report()
    # report() asks whether the output is well-formed. parse_audit asks whether it is
    # RIGHT: five checks comparing the annotation against the RTL, none of which reads
    # the reference. Baseline on gen2 was 92 prose/edge contradictions and 237 hidden
    # hops; those are what the new rules target.
    print()
    parse_audit.audit()
    # The re-parse exists to move GOVERNING-EDGE COVERAGE. The previous parse sat at
    # 51%, and the separation that signal carries between true and false dotted-signal
    # assets fell from +52 points in the RTL to +31 in that parse. Compare directly:
    from pathlib import Path as _P
    if _P('parsed_v3_tuning18_gen2').exists():
        print(chr(10) + "=== against gen2 (the previous parse) ===")
        parse_v3.compare_parses('parsed_v3_tuning18_gen2')
elif VERSION == "v2rp":
    import parse_roles
    parse_roles.role_report()
else:
    print(f"{VERSION}: no annotated parse for this version")

module                      elems    bad    bad%  isolated
neorv32_boot_rom               21      0    0.0%         9
neorv32_bus                   870      3    0.3%         0
neorv32_cache                  86      1    1.2%         0
neorv32_cpu                   115      0    0.0%         0
neorv32_cpu_cp_cfu             29      0    0.0%         3
neorv32_cpu_cp_muldiv          70      3    4.3%        35
neorv32_cpu_pmp                62      0    0.0%        30
neorv32_debug_dtm              42      0    0.0%         0
neorv32_fifo                   27      0    0.0%         0
neorv32_hwspinlock             21      0    0.0%        10
neorv32_imem                   27      2    7.4%        10
neorv32_package                 0      0    0.0%         0
neorv32_spi                    66      2    3.0%        10
neorv32_sys                    17      0    0.0%         0
neorv32_trng                   56      3    5.4%         8
neorv32_twi                    65      1    1.5%        

# Stage B: asset generation

In [30]:
# ===================== Stage B: version registry + generation =====================
import importlib, prompts_v2, rtl_parse
import icl_examples_01b, icl_examples_01b_nosum, icl_examples_01b_noparse
import icl_examples_01b_ot
import diagnose_step1 as DG
# The ICL chain is reloaded in dependency order: _noparse is built by transforming
# _nosum, which is built from _01b, so reloading only the leaf would leave it holding
# text derived from a stale parent.
for m in (prompts_v2, rtl_parse, icl_examples_01b, icl_examples_01b_nosum,
          icl_examples_01b_noparse, icl_examples_01b_ot, DG):
    importlib.reload(m)

# Register one (core prompt, ICL examples) pair per version. To add a version:
# drop a new module next to these, import it above, add a line here, set VERSION
# in the config cell. Splice by CONCATENATION -- the examples contain literal JSON
# braces, so an f-string would need every brace escaped.
PROMPT_VERSIONS = {
    # ---- v2 BASELINE -------------------------------------------------------------
    # prompts_v2.ASSET_V2_BASE = the supplied draft (preserved byte-for-byte as
    # ASSET_V2_DRAFT) + correction D (closed-set parity) + correction L (de-leak).
    # ICL is the P-2 block: no summary blocks, no PARSED JSON blocks, both case studies
    # and both RTL bodies intact.
    "v2": (prompts_v2.ASSET_V2_BASE,
           icl_examples_01b_noparse.ICL_ASSET_EXAMPLES_01B_NOPARSE),
    # ---- S-1 · container vs content ----------------------------------------------
    # Same ICL block as v2; the CORE gains ONE bullet in RULES. A structure that stores or
    # carries data is an asset only when what it holds is itself an asset. Targets the 24.2%
    # of false positives that are fields of a record holding NO ground-truth asset -- buffers
    # and protocol sub-blocks. Ceiling: emit 412.7 -> 337.7, precision 0.251 -> 0.304.
    # The distinction is the reference's own: trng's fifo fields are assets (it holds
    # entropy), twi's/uart's/spi's are not (ordinary traffic); key_mem is an asset, the
    # cipher's internals are not.
    "v2s1": (prompts_v2.ASSET_V2_S1,
             icl_examples_01b_noparse.ICL_ASSET_EXAMPLES_01B_NOPARSE),
    # ---- R-8 · revert the A-08 per-concept closure sweep --------------------------
    # Same ICL block as v2; the CORE loses ONE block and gains nothing. The only arm that
    # removes rather than adds, so nothing design-specific can leak through it. When A-08
    # was ADDED in v1 (v01c5 -> v01c6) it took emissions 331.2 -> 428.0 and precision
    # 0.285 -> 0.229; under a precision objective that trade inverts. Also the direct test
    # of whether the sweep is cancelling restrictive rules -- S-1 came back null while
    # elements-per-concept ROSE 4.49 -> 4.69.
    "v2r8": (prompts_v2.ASSET_V2_R8,
             icl_examples_01b_noparse.ICL_ASSET_EXAMPLES_01B_NOPARSE),
    # ---- X-3 · third worked case study (OpenTitan gpio) ---------------------------
    # CORE IS UNCHANGED -- byte-identical to v2. Only the ICL block moves, so this is one
    # proposition: does a composition-matched example change what the model binds?
    # The two existing case studies bind 18 of 18 assets to PORTS; our reference is 43%
    # port / 56% internal. OpenTitan gpio was picked by surveying the golden lists of all
    # 21 IPs in the LAsset repo: only 3 have any internal-signal assets and it is the only
    # one where they dominate (3 of 5: reg_we, reg_re, reg_error).
    # The two GPIO examples were checked for contradiction BEFORE building and agree: both
    # name the interface write-enable an asset (a PORT in omsp_gpio, an INTERNAL signal in
    # OpenTitan) and both reject the per-register decoded strobes.
    "v2x3": (prompts_v2.ASSET_V2_BASE,
             icl_examples_01b_ot.ICL_ASSET_EXAMPLES_01B_OT),
    # ---- ARMS --------------------------------------------------------------------
    # Add one line per arm, keys ALPHANUMERIC ONLY (v2e1, not v2_e1 -- see the config cell).
    # Each arm is CURRENT_BASELINE plus exactly ONE change, registered in ABLATION_LOG_V2.md
    # with an Expect and a decision rule BEFORE it runs. None is built yet.
    "v2x3r8": (prompts_v2.ASSET_V2_R8,
             icl_examples_01b_ot.ICL_ASSET_EXAMPLES_01B_OT),
    # S-3 - routed traffic belongs to the fabric, not to every stop along it.
    # ONE change from v2: a single RULES bullet added to the core. The ICL is the
    # BASELINE block, not the OT block -- pairing this with the X-3 ICL would make it
    # two changes, and pairing it with ASSET_V2_R8 would confound a selective criterion
    # with a fan-out reduction, which is exactly what made X-3R-8 uninterpretable.
    "v2s3": (prompts_v2.ASSET_V2_S3,
             icl_examples_01b_noparse.ICL_ASSET_EXAMPLES_01B_NOPARSE),
    # SEC - line-6 expansion: demote influencers to a per-primary secondary array.
    # FIRST ARM TO TOUCH ALGORITHM 1 LINE 6. Core gains STEP 3 (SAIF's primary/secondary
    # definition + its two worked examples + a demotion test) and ONE output-contract
    # field. ICL is the BASELINE block. Oracle headroom, measured against LAsset's own
    # secondary lists: the "secondary-ONLY" population is 133.3 FP/rep against 4.0 TP/rep
    # (33:1); demoting it gives precision 0.249 -> 0.358 (+0.110), recall -0.036.
    # SCORING IS UNCHANGED - the scorer reads "Assets" only and ignores the new field;
    # verified by injecting secondary arrays into a v2 run and re-scoring (identical).
    "v2sec": (prompts_v2.ASSET_V2_SEC,
              icl_examples_01b_noparse.ICL_ASSET_EXAMPLES_01B_NOPARSE),
    # RP - restore the parsed closed set, now carrying functional roles.
    # THE FIRST ARM TO CHANGE THE INPUT REGIME RATHER THAN THE PROMPT TEXT. Undoes
    # correction D and hands line 5 the parsed ports/signals again, annotated by
    # prompts_parse_v2 with roles / relationship / evidence. The core gains the restored
    # INPUTS declaration, the two RULES bullets that name the JSON fields, and ONE new
    # bullet declaring the annotations evidence rather than verdicts -- without that
    # bullet the natural reading of a role like SECRET_DATA_STORAGE is "emit this", which
    # would move STEP 1's C/I/A decision into the parser.
    #
    # ICL IS THE BASELINE BLOCK AND STAYS THAT WAY. It has both PARSED spans removed, so
    # the worked examples reason from RTL alone while the live message carries a parsed
    # list. That asymmetry is the arm's chief risk and is REGISTERED, not accidental:
    # pairing this with a parsed-carrying ICL would be two changes, which is what made
    # X-3R-8 uninterpretable. _BLOCK_MARKERS below does NOT catch it -- that assertion
    # only fires when a block is dropped from the message while the prompt declares it,
    # and this is the reverse direction.
    #
    # Prior: v1's P-2 removed the parsed blocks and was NULL on every paired delta
    # (P 0.210 -> 0.243, recall 0.897 -> 0.886). That was a name+function list; this arm's
    # claim is that the ANNOTATION is what would have mattered. A null is a real result.
    "v2rp": (prompts_v2.ASSET_V2_RP,
             icl_examples_01b_noparse.ICL_ASSET_EXAMPLES_01B_NOPARSE),
    # P3 - the same input regime as RP, a different ANNOTATION.
    # RP receives `roles` from a 152-label closed taxonomy. P3 receives
    # `functionality` / `role` / `relationship` / `evidence` from prompts_parse_v3,
    # where role is open prose derived from an occurrence profile and the annotator
    # is handed no vocabulary whatsoever.
    #
    # A SEPARATE ARM RATHER THAN AN EDIT TO RP, because assets_tuning18_v2rp_r0..r2
    # exist and their _run_meta.json records the composed sha of ASSET_V2_RP.
    # Rewriting that string would leave three paid runs attributed to code that no
    # longer exists. Keeping both also buys the comparison for free: RP and P3 differ
    # in exactly one thing -- the vocabulary the PARSER was given -- so the pair
    # answers whether a closed enum or open prose transfers more to the asset stage.
    #
    # The core differs from RP in its INPUTS field list and in ONE rule: v3 emits no
    # security vocabulary in either direction, so the rule states that the neutral
    # tone of every annotation carries no information. Without it a model reading
    # fifteen hundred bland descriptions can read the blandness as a verdict.
    "v2p3": (prompts_v2.ASSET_V2_P3,
             icl_examples_01b_noparse.ICL_ASSET_EXAMPLES_01B_NOPARSE),
    # C1 - teach the asset stage how to READ the v3 annotations.
    # ONE ADDITION TO P3. P3 says what the four fields are and are not; it never says how
    # to use them, so the model improvises a reading of a structured input it has not seen
    # before. C1 inserts a READING THE ANNOTATIONS block ahead of STEP 1: which of the four
    # fields bears on which of the C/I/A/U questions, and what may not be inferred.
    #
    # NO ASSET CRITERIA ARE ADDED. The definition of a primary asset is untouched, so a
    # delta against v2p3 is attributable to the reading procedure alone.
    #
    # The lever against over-emission is "A COUPLING IS NOT A REASON". Every element in a
    # working design is coupled to something and most carry several edges; handed 1689
    # annotated elements a model can read connectedness as significance and name most of
    # the file. The block says an edge gives mechanism and never the reason, and that the
    # Justification must still argue from the module's purpose.
    "v2p3c1": (prompts_v2.ASSET_V2_P3C1,
               icl_examples_01b_noparse.ICL_ASSET_EXAMPLES_01B_NOPARSE),
}

_core, _icl = PROMPT_VERSIONS[VERSION]
ASSET_PRIMARY_SYSTEM = _core + "\n\n" + _icl

# Derived, never hand-maintained: if the composed prompt asks for the array, the run loop
# demands it back. A hand-kept list of "versions that emit concepts" is one edit away from
# the v01c4 failure, where a silent tolerance let 90 module-repeats pass while emitting none.
EXPECT_CONCEPTS = "ConceptualAssets" in ASSET_PRIMARY_SYSTEM

print(f"{VERSION}: core {len(_core)} + ICL {len(_icl)} = {len(ASSET_PRIMARY_SYSTEM)} chars"
      + ("   [expects ConceptualAssets]" if EXPECT_CONCEPTS else ""))


# Which user-message blocks a version receives. Absent version -> everything, so no
# existing version changes behaviour. The INPUT ABLATION family (P-1 / P-2' / P-3) asks
# whether line 5's over-emission is driven by input volume: it gets ~34.5k tokens per call
# (summary 11%, parsed ports 42%, parsed signals 16%, RTL 31%) and emits 428 assets for 111
# references, while LASP feeds a compact RTL abstraction because it cannot fit the RTL.
#
# Dropping a block here is only HALF the arm -- the core prompt declares its inputs and
# refers back to them, so each ablated version also pairs with a core variant that removes
# those references (prompts.ASSET_PRIMARY_CORE_P1 and friends). Leaving them would tell the
# model to ground in something absent, which is a confound rather than a removal.
INPUT_BLOCKS = {
    "v2": {"summary": False, "parsed": False},      # comment-stripped RTL only
    "v2s1": {"summary": False, "parsed": False},    # S-1: same input regime as v2
    "v2r8": {"summary": False, "parsed": False},    # R-8: same input regime as v2
    "v2x3": {"summary": False, "parsed": False},    # X-3: same input regime as v2
    "v2x3r8": {"summary": False, "parsed": False},  # X-3R-8: same input regime as v2
    "v2s3": {"summary": False, "parsed": False},    # S-3: same input regime as v2
    "v2sec": {"summary": False, "parsed": False},   # SEC: same input regime as v2
    # RP is the ONLY version that keeps the parsed blocks. summary stays False so every
    # other input is identical to v2. RETRY_ON_VALIDATION is derived from this and flips
    # back to True on its own -- correct here, and not the leak it would be under P-2:
    # the corrective retry selects on the parsed closed set, which this version puts IN
    # the prompt rather than withholding.
    "v2rp": {"summary": False, "parsed": True},     # RP: the parsed blocks come back
    "v2p3": {"summary": False, "parsed": True},     # P3: same regime, v3 annotation
    "v2p3c1": {"summary": False, "parsed": True},   # C1: same input, taught how to read it
}
# _ALL_BLOCKS is the DEFAULT, INPUT_BLOCKS is a sparse OVERRIDE. They do not conflict:
# build_asset_user merges {**_ALL_BLOCKS, **override} and later keys win, so v01c6p1
# resolves to summary=False with everything else left True. A version absent from
# INPUT_BLOCKS gets everything, so no existing version changes behaviour.
_ALL_BLOCKS = {"summary": True, "parsed": True, "rtl": True, "function": True}

# Two ways this could go wrong silently, both now checked at import.
#
# (1) A TYPO in an override key. {"summry": False} merges cleanly, leaves summary=True, and
#     the block is NOT dropped -- the arm would run as its own baseline and look like a
#     perfect null.
# (2) DRIFT between the dropped block and the prompt. If a version drops a block while its
#     core or ICL still declares it, the model is told to use something absent. That is a
#     confound, not an ablation -- and it is exactly what nearly wrecked P-1, whose ICL
#     examples still cited the summary by section after the core had stopped declaring it.
_BLOCK_MARKERS = {                       # (marker in core, marker in ICL examples)
    "summary":  ("TECHNICAL SUMMARY", "=== TECHNICAL SUMMARY ==="),
    "parsed":   ("PARSED I/O PORTS", "=== PARSED I/O PORTS (JSON) ==="),
    "rtl":      ("(4) RTL", "=== RTL ==="),
    "function": ("type, function", '"function":'),
}
for _v, _b in INPUT_BLOCKS.items():
    _unknown = set(_b) - set(_ALL_BLOCKS)
    assert not _unknown, (f"INPUT_BLOCKS[{_v!r}]: unknown block(s) {sorted(_unknown)} -- "
                          f"valid keys are {sorted(_ALL_BLOCKS)}. A typo here drops nothing.")
    assert _v in PROMPT_VERSIONS, f"INPUT_BLOCKS[{_v!r}] has no PROMPT_VERSIONS entry"
    _vcore, _vicl = PROMPT_VERSIONS[_v]
    for _k, _keep in _b.items():
        if _keep:
            continue
        _cm, _im = _BLOCK_MARKERS[_k]
        assert _cm not in _vcore, (f"{_v} drops '{_k}' from the user message but its CORE "
                                  f"still declares it ({_cm!r}) -- confound, not ablation")
        assert _im not in _vicl, (f"{_v} drops '{_k}' from the user message but its ICL "
                                 f"examples still demonstrate it ({_im!r}) -- the model would "
                                 f"be shown a procedure it cannot carry out")

# Also derived, and for the same reason. validate_primary checks every emitted name
# against the PARSED closed set, and the run loop reacts to a failure by regenerating
# once and keeping whichever attempt has fewer issues. For every version that receives
# the parsed blocks that retry is inert -- v01c6 and v01c6p1 both sit at 0 ungrounded
# names across all 90 module-repeats, so it has nothing to fire on.
#
# Under P-2 it would fire constantly, and that is not a cost, it is a LEAK: choosing
# the sample that agrees better with the parsed list is selection ON the closed set,
# using exactly the information the arm withholds from the prompt. The measurement
# would be of a two-sample filter, not of the model. So a version that drops the
# parsed blocks validates and REPORTS but does not regenerate.
#
# Derived from INPUT_BLOCKS rather than hand-listed: a hand-kept set of "versions
# that skip the retry" is one forgotten edit away from silently re-introducing the
# leak, which is the v01c4 failure mode in a different costume.
#
# DEFINED HERE, NOT BESIDE EXPECT_CONCEPTS. It reads INPUT_BLOCKS, which is declared
# above -- placing it earlier raises NameError in a fresh kernel and, in a kernel that
# has run this cell before, silently reads the PREVIOUS INPUT_BLOCKS and resolves to
# True. That failure leaves no trace in the output.
RETRY_ON_VALIDATION = INPUT_BLOCKS.get(VERSION, {}).get("parsed", True)
if not RETRY_ON_VALIDATION:
    print(f"{VERSION}: validation is REPORT-ONLY (no corrective regeneration) -- "
          f"this version does not receive the parsed closed set")


def build_asset_user(stem, summary, parsed, rtl, blocks=None):
    b = {**_ALL_BLOCKS, **(blocks or {})}
    ports, signals = parsed["ports"], parsed["signals"]
    if not b["function"]:                   # P-2': keep the closed set, drop the annotation
        ports = [{k: v for k, v in e.items() if k != "function"} for e in ports]
        signals = [{k: v for k, v in e.items() if k != "function"} for e in signals]
    parts = [f"TARGET IP MODULE: {stem}\n\n"]
    if b["summary"]:
        parts.append(f"=== TECHNICAL SUMMARY ===\n{summary}\n\n")
    if b["parsed"]:
        parts.append(f"=== PARSED I/O PORTS (JSON) ===\n{json.dumps(ports, indent=1)}\n\n")
        parts.append(f"=== PARSED INTERNAL SIGNALS (JSON) ===\n{json.dumps(signals, indent=1)}\n\n")
    if b["rtl"]:
        parts.append(f"=== RTL ===\n{rtl}\n\n")
    parts.append(f"Identify the primary security assets for '{stem}' and return the JSON object per the contract.")
    return "".join(parts)

def generate_assets(stem, path, parsed, out_dir, overwrite=False):
    """Cache ONLY parse-valid results.

    An unparseable response raises instead of being written, so a failed call can never
    masquerade as a genuine "this module has no assets" verdict -- which would silently
    become a free precision win for whichever prompt version happened to fail.
    """
    outp = Path(out_dir) / f"{stem}.json"
    if outp.exists() and not overwrite:
        return json.loads(outp.read_text(encoding="utf-8"))
    # Read the summary ONLY if this version actually receives it. v1 read it
    # unconditionally and let build_asset_user discard it, which made every v2 run
    # depend on summaries_tuning18/*.md for data that is never placed in the prompt.
    blocks = {**_ALL_BLOCKS, **INPUT_BLOCKS.get(VERSION, {})}
    summary = ((OUT_DIR / f"{stem}.md").read_text(encoding="utf-8")
               if blocks["summary"] else "")
    # Comments are stripped for the ASSET stage only. The asset prompt takes its
    # semantics from the technical summary and the parsed elements' `function` fields,
    # not from in-source comments -- so removing them drops ~30-60% of the RTL text
    # without losing anything the parser or the prompt relies on (parse_rtl_file returns
    # an identical closed set either way). Do NOT strip for the port/signal ANNOTATION
    # stage: that prompt is explicitly told to use in-source comments.
    rtl = rtl_parse.strip_comments(
        Path(path).read_text(encoding="utf-8", errors="ignore"),
        vhdl=Path(path).suffix.lower() in (".vhd", ".vhdl"))
    user = build_asset_user(stem, summary, parsed, rtl, blocks)
    # max_output_tokens is a CEILING, not a reservation: you are billed for tokens the
    # model actually generates, so a high cap costs nothing on calls that finish early.
    # A single generous budget beats an escalation ladder, whose truncated first attempt
    # was discarded and paid for anyway (~12k wasted output tokens per escalation; at
    # effort="high" reasoning is ~83% of output, leaving little room for the JSON).
    txt = client.complete(ASSET_PRIMARY_SYSTEM, user, max_tokens=65536, effort="high")
    # Keep the response VERBATIM, before parsing, in a subdirectory that load_run() cannot
    # see (it globs "*.json" non-recursively, so a nested folder is invisible to scoring).
    # Costs disk and nothing else. v01c4's entire concept layer was destroyed because only
    # the reshaped result was ever written -- 90 calls, unrecoverable. Written pre-parse so
    # a malformed response is captured too, which is precisely when the original matters.
    raw_dir = Path(out_dir) / "_raw"
    raw_dir.mkdir(exist_ok=True)
    (raw_dir / f"{stem}.json").write_text(txt, encoding="utf-8")
    obj = _loads(txt)
    if not (isinstance(obj, dict) and isinstance(obj.get("Assets"), list)):
        raise RuntimeError(f"{stem}: unparseable asset response -- nothing cached")
    # PRESERVE EVERY TOP-LEVEL KEY. This used to be {"IP": stem, "Assets": obj["Assets"]},
    # which silently deleted anything else the model returned before it reached disk. The
    # T-1 arm asked for a "ConceptualAssets" array, the model emitted it, and this line
    # threw it away in all 90 files of v01c4 -- which was then read as the model refusing
    # to follow the contract. It also produced the false clue that broke the diagnosis:
    # obj["Assets"] passes through wholesale, so fields nested INSIDE each asset survived
    # while top-level keys vanished, and that asymmetry looked like evidence about the
    # model when it was evidence about this line.
    # Only IP is normalised, which is all the rebuild was ever for -- it must match the
    # filename. Unexpected keys are now cached rather than dropped, so the validators can
    # flag them instead of them disappearing without trace.
    result = {"IP": stem, **{k: v for k, v in obj.items() if k != "IP"}}
    outp.write_text(json.dumps(result, indent=2), encoding="utf-8")
    return result

def _lev(a, b, cap=6):
    """Levenshtein distance, with an early exit once it exceeds `cap`.

    Only used to answer "is this a near-miss or an invention?", so a distance of 7 and a
    distance of 40 are the same answer and there is no reason to compute either exactly.
    """
    if a == b:
        return 0
    if abs(len(a) - len(b)) > cap:
        return cap + 1
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        if min(cur) > cap:
            return cap + 1
        prev = cur
    return prev[-1]


def validate_primary(result, parsed, rtl_text=""):
    """-> [record], one per problem. Keyed by (entity, name): duplicate names across a
    file's entities (e.g. clk_i in neorv32_trng/neoTRNG/neoTRNG_cell) must not yield false
    Entity mismatches.

    RECORDS, NOT STRINGS. Every version through v01c6p1 receives the parsed closed set and
    sits at 0 ungrounded names across all 90 module-repeats, so the string form cost
    nothing. P-2 withholds that list and asks the model to read the declarations out of the
    RTL, which makes an ungrounded name the arm's most informative output -- and it has
    three quite different causes that a string cannot distinguish:

      in_rtl_base True,  distance small  -> the element is real; our regex parser missed it,
                                            or the model wrote a variant spelling
      in_rtl_base True,  distance large  -> real identifier, wrong granularity or wrong name
      in_rtl_base False                  -> invented

    Only the first is a finding about rtl_parse rather than about the model, and separating
    them is the whole point of keeping this data.

    `rtl_text` is optional so the function still works from analysis scripts that have no
    source to hand; the RTL-presence fields are then None rather than False, because
    "not checked" and "checked, absent" must never look alike.
    """
    elems = parsed["ports"] + parsed["signals"]
    keys  = {(e["entity"], e["name"]) for e in elems}
    ents  = {}
    for e in elems:
        ents.setdefault(e["name"], set()).add(e["entity"])
    names = list(ents)
    OBJ = {"Confidentiality", "Integrity", "Availability"}
    out = []
    for a in result.get("Assets", []):
        r, en = a.get("Asset RTL"), a.get("Entity")
        obj = a.get("Security Objective")
        base = {"name": r, "entity_claimed": en, "concept": a.get("Concept"),
                "objective": obj}
        if r not in ents:
            tok = str(r).split(".")[0] if r else ""
            if rtl_text and tok:
                full = bool(re.search(rf"\b{re.escape(str(r))}\b", rtl_text))
                bse  = bool(re.search(rf"\b{re.escape(tok)}\b", rtl_text))
            else:
                full = bse = None
            near, dist = min(((n, _lev(str(r), n)) for n in names),
                             key=lambda t: t[1]) if names else (None, None)
            out.append({**base, "kind": "ungrounded", "in_rtl_full": full,
                        "in_rtl_base": bse, "nearest": near, "distance": dist,
                        "detail": f"ungrounded Asset RTL '{r}'"})
        elif (en, r) not in keys:
            out.append({**base, "kind": "entity_mismatch", "entity_actual": sorted(ents[r]),
                        "detail": f"Entity mismatch for '{r}': '{en}' not in {sorted(ents[r])}"})
        if obj not in OBJ:
            out.append({**base, "kind": "bad_objective",
                        "detail": f"bad objective for '{r}': '{obj}'"})
    return out


v2p3c1: core 11885 + ICL 62848 = 74735 chars   [expects ConceptualAssets]


In [31]:
# ============ Which closed set Stage B receives ============
# Each annotated version receives ITS OWN parse; every other version receives the v1 one.
# An explicit dispatch rather than a re-pointed parse_module, so one glance at this cell
# shows which closed set is in play.
#
# All parses contain THE SAME ELEMENTS by construction — parse_roles copies the closed set
# from parsed_tuning18/ and parse_v3 re-derives it with the same regex and then asserts it
# matches — so validate_primary and the scorer behave identically whichever is loaded. Only
# the annotation fields differ, which is exactly what the arm is testing.
#
# THE FIELD-SIGNATURE ASSERTION IS THE ANTI-CONTAMINATION CHECK. The two regimes are
# distinguishable by their per-element keys: v3 writes `role` (open prose), the taxonomy
# parse wrote `roles` (enum labels). Loading the wrong directory would otherwise be silent
# and would only surface as an inexplicable emission delta after the run was paid for.
#
# MUST RUN AFTER the annotated-parse cell and BEFORE the run loop below.
if VERSION in ("v2p3", "v2p3c1"):
    import importlib, parse_v3
    importlib.reload(parse_v3)
    parsed_all = parse_v3.load_all(modules)
    assert parsed_all, (f"{VERSION}: parsed_v3_tuning18/ is empty -- run the annotated-parse cell "
                        "first, or Stage B would silently fall back to the v1 parse")
    _keys = {k for p in parsed_all.values()
             for e in p["ports"][:1] + p["signals"][:1] for k in e}
    assert {"function", "functionality", "role", "relationship", "evidence"} <= _keys, \
        f"v2p3: loaded parse is missing v3 fields; got {sorted(_keys)}"
    assert "roles" not in _keys, \
        f"{VERSION}: loaded parse carries the v2rp taxonomy field `roles` -- wrong directory"
    _n = sum(len(p["ports"]) + len(p["signals"]) for p in parsed_all.values())
    print(f"{VERSION}: v3 parse, {len(parsed_all)} modules, {_n} elements")
    print(f"   per-element fields: {sorted(_keys)}")

elif VERSION == "v2rp":
    import importlib, parse_roles
    importlib.reload(parse_roles)
    parsed_all = parse_roles.load_all(modules)
    assert parsed_all, "v2rp: parsed_roles_tuning18/ is empty -- run the parse cell first"
    _keys = {k for p in parsed_all.values()
             for e in p["ports"][:1] + p["signals"][:1] for k in e}
    assert "roles" in _keys, f"v2rp: loaded parse is not the taxonomy one; got {sorted(_keys)}"
    print(f"v2rp: taxonomy parse, {len(parsed_all)} modules")

else:
    print(f"{VERSION}: v1 parse (mechanical closed set, short `function` only)")

   normalised on load (files unchanged): target_dropped=2
v2p3c1: v3 parse, 18 modules, 1689 elements
   per-element fields: ['dir', 'entity', 'evidence', 'function', 'functionality', 'kind', 'name', 'relationship', 'role', 'type']


In [32]:
# ===== Stage B run loop: REPEATS x (generate -> validate -> one corrective retry) =====
# Each repeat re-runs the SAME prompt into its own directory. The spread across repeats
# is the noise floor; read it before believing any difference between versions.
# Re-running this cell is cheap: a module already cached in a repeat dir is not regenerated.
#
# Modules within a repeat run CONCURRENTLY; repeats stay sequential. Lower MAX_WORKERS if
# the API starts rate-limiting -- the client already retries with exponential backoff, but
# retries cost wall clock, so throttling here is cheaper than being throttled there.
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import Counter
MAX_WORKERS = 6
# The parse this run reads, recorded alongside the prompt sha. Empty for RTL-only arms.
PARSE_FP = ""
if VERSION in ("v2p3", "v2p3c1"):
    import parse_v3
    PARSE_FP = parse_v3.parse_fingerprint()
    print(f"parse fingerprint: {PARSE_FP[:16]}")

for rep in range(REPEATS):
    out_dir = asset_dir(VERSION, rep, create=True)
    # Provenance: which prompt actually produced this directory. Underscore-prefixed so
    # eval_assets.load_run() skips it rather than reading it as a module.
    #
    # Written ONCE per directory, and never blindly. generate_assets() returns any module
    # already cached, so re-running into an existing directory keeps the old files and
    # generates only what is missing. If the prompt changed in between, that directory then
    # holds output from TWO prompts -- and overwriting the meta would stamp it with the new
    # sha and hide the mix completely. So:
    #   sha matches  -> leave the existing file alone; its timestamp is the true one
    #   sha differs  -> refuse; the directory can no longer be attributed to one prompt
    # (An earlier unconditional write clobbered v01c3 r0-r2's timestamps during the n=5
    #  extension. Harmless there only because the sha was unchanged.)
    meta_p = out_dir / "_run_meta.json"
    sha = hashlib.sha256(ASSET_PRIMARY_SYSTEM.encode()).hexdigest()
    if meta_p.exists():
        old = json.loads(meta_p.read_text())
        if PARSE_FP and old.get("parse_fingerprint") not in (None, PARSE_FP):
            raise RuntimeError(
                f"{out_dir} was generated from parse {str(old.get('parse_fingerprint'))[:12]} "
                f"but the current parse is {PARSE_FP[:12]}. The asset prompt is unchanged, so "
                f"this directory would be served from cache and the re-parse would have no "
                f"effect. Move the directory aside, or use a new VERSION.")
        if old.get("system_prompt_sha256") != sha:
            raise RuntimeError(
                f"{out_dir} holds assets from prompt {str(old.get('system_prompt_sha256'))[:12]} "
                f"but the current prompt is {sha[:12]}. Re-running here would mix two prompts "
                f"in one run directory. Delete the directory, or use a new VERSION.")
    else:
        meta_p.write_text(json.dumps({
            "version": VERSION, "repeat": rep, "model": GEN_MODEL, "effort": "high",
            "system_prompt_sha256": sha,
            "system_prompt_chars": len(ASSET_PRIMARY_SYSTEM),
            # For a parsed-input arm the PARSE is half of what the model reads, and the
            # asset sha does not move when it changes. Without this, two runs of one arm on
            # two different parses are indistinguishable afterwards -- and because the sha
            # still matches, the check above accepts the old directory and generate_assets
            # serves every module from cache, so a re-run after a re-parse is a silent
            # no-op that looks like a completed run.
            "parse_fingerprint": PARSE_FP,
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        }, indent=2))

    unresolved = {}
    print(f"\n===== {VERSION} repeat {rep}  ->  {out_dir} =====")

    def _check(stem, res, path):
        """Every validation record for one result: structural + concept-contract.

        The RTL is re-read here rather than threaded down from generate_assets, so that a
        module served from cache is validated against the same source a freshly generated
        one is. It is a file read and a regex per module per repeat.
        """
        rtl_txt = rtl_parse.strip_comments(
            Path(path).read_text(encoding="utf-8", errors="ignore"),
            vhdl=Path(path).suffix.lower() in (".vhd", ".vhdl"))
        return (validate_primary(res, parsed_all[stem], rtl_txt)
                + [{"kind": "concept_contract", "name": None, "detail": s}
                   for s in DG.validate_concepts(res, EXPECT_CONCEPTS)])

    def _one(stem, path):
        """generate -> validate -> (only if RETRY_ON_VALIDATION) one corrective retry."""
        res = generate_assets(stem, path, parsed_all[stem], out_dir)
        recs = _check(stem, res, path)
        note = ""
        if recs and RETRY_ON_VALIDATION:
            note = f"[warn] {stem}: {len(recs)} issue(s) -> one regeneration\n        " \
                   + "\n        ".join(r["detail"] for r in recs)
            res2 = generate_assets(stem, path, parsed_all[stem], out_dir, overwrite=True)
            recs2 = _check(stem, res2, path)
            if len(recs2) <= len(recs):
                res, recs = res2, recs2
            else:   # first attempt was better -- restore it to the cache
                (out_dir / f"{stem}.json").write_text(json.dumps(res, indent=2),
                                                 encoding="utf-8")
        return res, recs, note

    # The 18 modules are independent, I/O-bound calls that each write their own file, so
    # they parallelise cleanly. Wall clock was ~3.9 min/call sequentially; the work is
    # almost entirely waiting on the API, not local compute (which is also why a GPU does
    # nothing here -- the model runs on OpenAI's side).
    # Results are collected and printed IN MODULE ORDER after the pool drains, so the log
    # stays diffable across runs instead of interleaving by completion time.
    done = {}
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futs = {pool.submit(_one, stem, path): stem for stem, path in modules}
        for fut in as_completed(futs):
            stem = futs[fut]
            try:
                done[stem] = fut.result()
            except Exception as e:
                done[stem] = (None, None, f"[err ] {stem:24s} {e}")

    # The report is WRITTEN, not just printed. It used to live in a local `unresolved`
    # dict that died with the cell, which was tolerable while every version sat at zero
    # issues. Underscore-prefixed so eval_assets.load_run() skips it -- it globs "*.json"
    # and ignores names starting with "_", so this cannot be scored as a module.
    report = {"version": VERSION, "repeat": rep, "model": GEN_MODEL,
              "system_prompt_sha256": sha,
              "retry_on_validation": RETRY_ON_VALIDATION,
              "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
              "modules": {}, "totals": {}}
    for stem, _path in modules:
        res, recs, note = done[stem]
        if note:
            print(note)
        if res is None:
            continue
        if recs:
            unresolved[stem] = [r["detail"] for r in recs]
        kinds = Counter(r["kind"] for r in recs)
        report["modules"][stem] = {
            "assets": len(res.get("Assets", [])),
            "concepts": len(res.get("ConceptualAssets", []) or []),
            "by_kind": dict(kinds),
            "issues": recs,
        }
        print(f"[ok  ] {stem:24s} {len(res['Assets']):3d} assets"
              + (f"   {dict(kinds)}" if recs else ""))

    tot = Counter()
    for _m in report["modules"].values():
        tot.update(_m["by_kind"])
        tot["assets"] += _m["assets"]
    report["totals"] = dict(tot)
    (out_dir / "_validation.json").write_text(json.dumps(report, indent=1), encoding="utf-8")

    _n = tot.get("assets", 0)
    print(f"  validation -> {out_dir / '_validation.json'}")
    print(f"  totals: {dict(tot)}"
          + (f"   ungrounded {tot.get('ungrounded', 0) / _n:.1%} of emissions" if _n else ""))
    print("  policy: " + ("REPORT-ONLY -- no asset was regenerated on a validation result"
                          if not RETRY_ON_VALIDATION else
                          f"retry ON -- unresolved: {unresolved or 'none'}"))

print("\n", client.cost())


parse fingerprint: 540765753f16a63d

===== v2p3c1 repeat 0  ->  assets_tuning18_v2p3c1_r0 =====
[ok  ] neorv32_boot_rom           6 assets
[warn] neorv32_bus: 4 issue(s) -> one regeneration
        asset 'device_req_o' -> unknown concept 'C7'
        asset 'host_rsp_o' -> unknown concept 'C7'
        asset 'port_sel' -> unknown concept 'C8'
        asset 'main_req.addr' -> unknown concept 'C8'
[ok  ] neorv32_bus               34 assets   {'concept_contract': 2}
[ok  ] neorv32_cache             50 assets
[ok  ] neorv32_cpu               51 assets
[ok  ] neorv32_cpu_cp_cfu        19 assets
[ok  ] neorv32_cpu_cp_muldiv     26 assets
[ok  ] neorv32_cpu_pmp           27 assets
[ok  ] neorv32_debug_dtm         27 assets
[ok  ] neorv32_fifo              25 assets
[ok  ] neorv32_hwspinlock         7 assets
[ok  ] neorv32_imem              11 assets
[warn] neorv32_package: 1 issue(s) -> one regeneration
        ConceptualAssets missing entirely -- the prompt asked for it
[ok  ] neorv32_package 

# Stage C: evaluation

Metric **M-2**, unchanged from v1 — a v2 number and a v1 number are directly comparable.

**Primary metric is precision.** Recall is the guard: an arm that raises precision but drops
recall more than 0.03 below its own baseline does not become the new baseline. Judged on the
point estimate, not the CI — at 15 modules the paired recall CI is about ±0.14, so a
CI-gated guard would never fire. Drop of 0.03–0.06 is indeterminate: re-run at n=5.

**F1 is reported second.** It stays flat when an arm merely emits less and rises when an arm
removes false positives without losing true ones, which is exactly the discrimination this
study needs.

In [12]:
# ===== Build the reference sets from the LAsset release (run once; re-run if the xlsx changes)
import importlib, gt_extract
importlib.reload(gt_extract)

LASSET_ROOT = Path(r"E:\jobs\ff\test\LAsset-Security-Assets")   # <-- the published results repo
GT_DIR      = Path("ground_truth")

gt_extract.main(LASSET_ROOT, GT_DIR)

# Sanity: the extracted counts must reconcile with the paper's own Statistics sheet
# (302 True Assets / 359 initial / 331 refined). A mismatch means the sheet layout
# changed and IP_LABEL_TO_STEM or split_elements() needs revisiting.
for f in sorted(GT_DIR.glob("*.json")):
    d = json.loads(f.read_text(encoding="utf-8"))
    n = sum(len(m["assets"]) for m in d["modules"].values())
    print(f"  {f.name:28s} {len(d['modules']):3d} modules, {n:4d} elements")

manual GT : 41 modules, 246 rows -> 302 elements -> ground_truth\manual_gt_neorv32.json
            (paper's Statistics sheet reports 302 True Assets)
lasset initial : 41 modules, 359 assets -> ground_truth\lasset_initial.json
lasset refined : 41 modules, 331 assets -> ground_truth\lasset_refined.json
lasset final   : empty, skipped
  lasset_initial.json           41 modules,  359 elements
  lasset_refined.json           41 modules,  331 elements
  manual_gt_neorv32.json        41 modules,  302 elements


In [13]:
# ===== Score one run of the current VERSION =====
import importlib, eval_assets as EV
importlib.reload(EV)

REFS = EV.load_refs(GT_DIR)          # {"gt", "paper", "paper_refined"}

# collect() finds the repeat directories, and falls back to a pre-repeats run directory.
_runs = EV.collect([VERSION])
res = EV.score(_runs[VERSION][0], REFS["gt"]) if VERSION in _runs else None
if res:
    EV.table(res, f"{VERSION} vs manual ground truth (first repeat)")
else:
    print(f"no generated assets for {VERSION} yet -- run Stage B first")


v2p3c1 vs manual ground truth (first repeat)
                        ref  emit   TP   FP   FN      P      R     F1    obj
----------------------------------------------------------------------------
bus                      11    35    9   26    2  0.257  0.818  0.391  1.000
cache                     6    30    1   29    5  0.033  0.167  0.056  1.000
cpu                      14    35   11   24    3  0.314  0.786  0.449  1.000
cpu_cp_cfu               11    18    9    9    2  0.500  0.818  0.621  0.444
cpu_cp_muldiv            12    25   12   13    0  0.480  1.000  0.649  0.750
cpu_pmp                   6    30    6   24    0  0.200  1.000  0.333  0.667
debug_dtm                 5    22    3   19    2  0.136  0.600  0.222  1.000
hwspinlock                2     8    2    6    0  0.250  1.000  0.400  1.000
imem                      3    11    3    8    0  0.273  1.000  0.429  0.333
spi                       8    43    7   36    1  0.163  0.875  0.275  1.000
sys                       2    

In [14]:
# RUN-ALL GUARD. Reads parsed_roles_tuning18/ and the v2rp runs.
if VERSION == "v2rp":
    # ====== RP DIAGNOSTIC — run AFTER v2rp finishes (CLOSED-TAXONOMY ARM ONLY) ======
    # Did emission track the parser's roles at all? This separates "the model used the
    # annotations" from "the model ignored them", and it is worth running whichever way the
    # aggregate lands.
    #
    # If emitted elements are no more likely to carry a positive lead role than a random
    # closed-set element, the annotations were decorative and the arm's result is about the name
    # list after all -- which would make it a replication of P-2's null rather than a new finding.
    import json as _json
    import prompts_parse_v2
    from pathlib import Path as _P

    _ROLES = {}
    for _f in sorted(_P("parsed_roles_tuning18").glob("*.json")):
        _d = _json.loads(_f.read_text(encoding="utf-8"))
        for _e in _d["ports"] + _d["signals"]:
            _ROLES[(_f.stem, _e["name"])] = _e["roles"][0]

    if not _ROLES:
        print("no parsed_roles_tuning18/ yet -- run the role-annotation cell")
    else:
        _POS = set(prompts_parse_v2.POSITIVE_ROLES)
        _base = sum(1 for _r in _ROLES.values() if _r in _POS) / len(_ROLES)
        for _v in ("v2", "v2rp"):
            _hit = _tot = 0
            for _dir in sorted(_P(".").glob(f"assets_tuning18_{_v}_r*")):
                for _f in _dir.glob("*.json"):
                    for _a in _json.loads(_f.read_text(encoding="utf-8")).get("Assets", []):
                        _r = _ROLES.get((_f.stem, _a.get("Asset RTL")))
                        if _r is None:
                            continue
                        _tot += 1
                        _hit += _r in _POS
            print(f"{_v:6s} emitted elements carrying a positive lead role: "
                  f"{_hit}/{_tot} = {100*_hit/_tot if _tot else 0:.1f}%")
        print(f"closed-set base rate: {100*_base:.1f}%   "
              f"(no lift over base = the annotations were decorative)")
else:
    print(f"skipped: this cell is for the v2rp arm, VERSION is {VERSION}")


skipped: this cell is for the v2rp arm, VERSION is v2p3c1


In [15]:
# RUN-ALL GUARD. Compares the annotation of emitted vs un-emitted elements for the current arm.
if VERSION in ("v2p3", "v2p3c1"):
    # ============ P3 DIAGNOSTIC — run AFTER v2p3 finishes ============
    # The RP diagnostic asks whether emitted elements carried a positive role. P3 has no labels,
    # so it cannot be asked that way. The equivalent question is whether the emitted elements
    # differ from the un-emitted ones in the annotation the parser produced.
    #
    # If they do not, the annotations were decorative and the arm is a replication of P-2's
    # null rather than a new finding -- the same conclusion, reached without labels.
    import json as _json
    from pathlib import Path as _P
    import statistics as _st

    _ANN = {}
    for _f in sorted(_P("parsed_v3_tuning18").glob("*.json")):
        _d = _json.loads(_f.read_text(encoding="utf-8"))
        for _e in _d["ports"] + _d["signals"]:
            _ANN[(_f.stem, _e["name"])] = _e

    if not _ANN:
        print("no parsed_v3_tuning18/ yet -- run the annotated-parse cell")
    else:
        def _profile(elems):
            """What the parser said about a set of elements, as three numbers."""
            if not elems:
                return None
            return (_st.mean(len(e.get("relationship") or []) for e in elems),
                    _st.mean(len(e.get("role") or []) for e in elems),
                    100 * sum(1 for e in elems
                              if any(str(r.get("type", "")).upper() == "GATES"
                                     for r in (e.get("relationship") or []))) / len(elems))

        _emitted = set()
        # VERSION, not a literal: under v2p3c1 a hardcoded "v2p3" would analyse the
        # other arm's runs and report its numbers as this one's.
        for _dir in sorted(_P(".").glob(f"assets_tuning18_{VERSION}_r*")):
            for _f in _dir.glob("*.json"):
                for _a in _json.loads(_f.read_text(encoding="utf-8")).get("Assets", []):
                    _emitted.add((_f.stem, _a.get("Asset RTL")))

        if not _emitted:
            print("no v2p3 runs yet -- set VERSION = 'v2p3' and run Stage B")
        else:
            _in = [v for k, v in _ANN.items() if k in _emitted]
            _out = [v for k, v in _ANN.items() if k not in _emitted]
            print(f"emitted {len(_in)} of {len(_ANN)} annotated elements\n")
            print(f"{'':22} {'edges/elem':>11} {'roles/elem':>11} {'has GATES %':>12}")
            for _lbl, _grp in (("emitted", _in), ("not emitted", _out)):
                _p = _profile(_grp)
                print(f"{_lbl:22} {_p[0]:11.2f} {_p[1]:11.2f} {_p[2]:11.1f}%")
            _a, _b = _profile(_in), _profile(_out)
            print("\nNo separation on any of the three = the annotations did not inform the "
                  "decision,\nand this arm is P-2's null reached by a different route.")
else:
    print(f"skipped: this cell is for the v3-parse arms, VERSION is {VERSION}")


emitted 522 of 1620 annotated elements

                        edges/elem  roles/elem  has GATES %
emitted                       1.98        2.38        21.8%
not emitted                   1.15        2.01         3.1%

No separation on any of the three = the annotations did not inform the decision,
and this arm is P-2's null reached by a different route.


In [18]:
# ===== ABLATION: every version, mean +/- sd over repeats =====
# Read the sd columns FIRST. They are the noise floor: a gap between two versions smaller
# than their sd is not measurable at this many repeats, however suggestive it looks.
importlib.reload(EV)

VERSIONS = ["v2", "v2s1", "v2r8", "v2x3", "v2x3r8", "v2s3", "v2sec", "v2p3c1"]  # add each arm here once it has run

# Each arm has a DESIGNATED baseline -- the version it differs from by one variable.
# Pairing everything against VERSIONS[0] silently produces multi-variable contrasts.
BASELINE = {"v2s1": "v2", "v2r8": "v2", "v2x3": "v2", "v2x3r8": "v2",
            "v2s3": "v2", "v2sec": "v2", "v2p3c1": "v2"}
# A new arm with no explicit entry pairs against the standing baseline. Explicit
# entries above always win, so promoting CURRENT_BASELINE can never silently
# re-point an old arm at a comparison it was not run against.
for _v in VERSIONS:
    if _v not in BASELINE and _v != CURRENT_BASELINE and _v != VERSIONS[0]:
        BASELINE[_v] = CURRENT_BASELINE

# DO NOT add "v01c6p1p2" to VERSIONS to get a "where we were vs where we are" row.
# ablate() pairs every version against VERSIONS[0], so it would print v2 vs v01c6p1p2 as
# though it were an arm. It is not: it bundles correction D, the L de-leak and a hand
# transcription. The log allows quoting that contrast in prose and never as a delta.

RUNS = EV.collect(VERSIONS)               # {version: [run per repeat]}; missing ones warn

if RUNS:
    # The paper's own generation stage, on exactly the modules every run has in common --
    # same algorithmic stage as ours, so this row is the target, not the ground truth.
    MODS = EV.common_modules(RUNS, REFS["gt"])
    paper_here = {m: v for m, v in REFS["paper"].items() if m in MODS}

    EV.ablate(RUNS, REFS["gt"], extra={"LAsset paper": paper_here})

    # PER-CLASS paired deltas are the primary result, not the aggregate. A-01 is why:
    # aggregate recall read +0.049 [-0.061, +0.166] -- "no effect" -- while port recall
    # rose +0.181 [+0.048, +0.302] and signal recall fell -0.157 [-0.229, -0.086]. Both
    # class effects clear zero; they cancel in the aggregate. Reporting only the aggregate
    # would have thrown away a resolvable result and called the arm a null.
    for v in VERSIONS:
        base = BASELINE.get(v)
        if v in RUNS and base in RUNS:
            n = min(len(RUNS[v]), len(RUNS[base]))
            note = "  [DIRECTIONAL ONLY -- fewer than 3 repeats, no usable sd]" if n < 3 else ""
            print(f"\n--- {v} vs its baseline {base}{note}")
            EV.paired_classes(RUNS, base, v, REFS["gt"])
        elif v in RUNS and base:
            print(f"\n--- {v}: baseline {base!r} has no run yet, skipped")
else:
    print("no runs found -- run Stage B for at least one version first")


ablation vs manual ground truth
scored on 15 modules common to every run
version       n  emit   TP   FP   FN      P  recall     sd      F1     sd  near
-------------------------------------------------------------------------------
v2            3   413  103  310    8  0.251   0.925  0.021   0.394  0.032   0.0
v2s1          3   425  104  321    7  0.246   0.937  0.009   0.389  0.024   0.0
v2r8          5   375   99  275   12  0.265   0.895  0.023   0.409  0.012   0.0
v2x3          3   391  100  291   11  0.259   0.901  0.018   0.402  0.048   0.0
v2x3r8        3   320   95  225   16  0.296   0.853  0.055   0.439  0.015   0.0
v2s3          3   420  105  316    6  0.249   0.943  0.014   0.394  0.009   0.0
v2sec         3   246   70  177   41  0.287   0.628  0.023   0.392  0.030   0.0
v2p3c1        3   376   94  281   17  0.251   0.850  0.014   0.388  0.013   1.0
LAsset paper  1   137  101   36   10  0.737   0.910          0.815         0.0

  recall by element class                   por

In [ ]:
# RUN-ALL GUARD. Reads the v2sec runs.
if VERSION == "v2sec":
    # ===== SEC DECISION CELL -- run AFTER v2sec finishes =====
    # The hypothesis is that elements MOVE from the primary list into a secondary array.
    # The predicted failure (four-for-four in this study: hand this model a population and it
    # names from it) is that the primary list stays put and a secondary list is merely ADDED.
    # Check 1 decides that, and it decides the arm regardless of what the aggregate says.
    importlib.reload(EV)
    import json as _json, statistics as _st
    from collections import Counter as _Counter

    _A, _B = "v2", "v2sec"
    _R = EV.collect([_A, _B]); _REF = EV.load_refs()["gt"]; _CL = EV.load_closed()
    _M = EV.common_modules(_R, _REF)

    def _prim(v, i):
        return {(m, x[1]) for m in sorted(_M) for x in EV._as_run(_R[v][i])[m]}

    def _sec(v, i):
        out = set()
        for m in sorted(_M):
            d = _json.load(open(f"assets_tuning18_{v}_r{i}/{m}.json", encoding="utf-8"))
            for a in d.get("Assets", []):
                for s in (a.get("secondary") or []):
                    out.add((m, s))
        return out

    print("1. MOVEMENT OR ADDITION?  (the arm's primary mechanism check)")
    _pa = _st.mean(len(_prim(_A, i)) for i in range(len(_R[_A])))
    _pb = _st.mean(len(_prim(_B, i)) for i in range(len(_R[_B])))
    _sb = _st.mean(len(_sec(_B, i)) for i in range(len(_R[_B])))
    print(f"     v2    distinct primary elements/rep : {_pa:6.1f}")
    print(f"     v2sec distinct primary elements/rep : {_pb:6.1f}   ({_pb-_pa:+.1f})")
    print(f"     v2sec distinct SECONDARY elements   : {_sb:6.1f}")
    # did the elements that LEFT the primary list turn up as secondary?
    _union_a = set().union(*[_prim(_A, i) for i in range(len(_R[_A]))])
    _union_pb = set().union(*[_prim(_B, i) for i in range(len(_R[_B]))])
    _union_sb = set().union(*[_sec(_B, i) for i in range(len(_R[_B]))])
    _left = _union_a - _union_pb
    _moved = _left & _union_sb
    print(f"     elements present in v2 primary, absent from v2sec primary : {len(_left)}")
    print(f"       ...of which appear in a v2sec secondary array           : {len(_moved)} "
          f"({100*len(_moved)/len(_left) if _left else 0:.0f}%)   <- MOVEMENT")
    _overlap = _union_pb & _union_sb
    print(f"     CONTRACT VIOLATION - element BOTH primary and secondary in the same repeat:")
    for i in range(len(_R[_B])):
        _bad = _prim(_B, i) & _sec(_B, i)
        print(f"       r{i}: {len(_bad)}" + (f"   e.g. {sorted(_bad)[:3]}" if _bad else ""))
    print("     REJECT if the primary list did not shrink, or if <50% of departures moved.\n")

    # ---- 2. hard checks: what a loose reading of 'infrastructure' would demote ----
    print("2. HARD CHECKS (any loss => reject)")
    _TY = {}
    import pathlib as _pl
    for _f in sorted(_pl.Path("parsed_tuning18").glob("*.json")):
        _d = _json.loads(_f.read_text(encoding="utf-8"))
        for _e in _d.get("ports", []) + _d.get("signals", []):
            _TY[(_f.stem, _e["name"])] = (_e.get("type") or "").strip()
    _ctrl = [(m, a[0]) for m in sorted(_M) for a in _REF[m]
             if _TY.get((m, a[0].split(".")[0]), "") == "ctrl_t"]
    _ext = [("neorv32_uart", "uart_rxd_i"), ("neorv32_uart", "uart_txd_o"),
            ("neorv32_spi", "spi_dat_i"), ("neorv32_twi", "twi_sda_i")]
    # group C: FSM / arbitration state. SAIF's OWN definition names "states of an FSM" as an
    # INTANGIBLE SECONDARY asset, but our reference lists 7 such elements as PRIMARY. That is a
    # registered conflict between the literature definition and this ground truth, and these are
    # the elements a literal reading of STEP 3 will demote first.
    _fsm = [("neorv32_bus", "state"), ("neorv32_bus", "sel"), ("neorv32_bus", "stb"),
            ("neorv32_bus", "port_sel"), ("neorv32_cpu_cp_muldiv", "ctrl.state"),
            ("neorv32_hwspinlock", "sel"), ("neorv32_wdt", "cnt")]
    for _label, _items in (("control record (all 17)", _ctrl), ("crosses design boundary", _ext),
                           ("FSM / arbitration state", _fsm)):
        _fa = _fb = 0
        for m, el in _items:
            for i in range(len(_R[_A])):
                if (m, el) not in _prim(_A, i): _fa += 1
            for i in range(len(_R[_B])):
                if (m, el) not in _prim(_B, i): _fb += 1
        _na, _nb = len(_items)*len(_R[_A]), len(_items)*len(_R[_B])
        print(f"     {_label:26s} missing: v2 {_fa}/{_na}   v2sec {_fb}/{_nb}"
              + ("   <== LOST" if _fb/_nb > _fa/_na else ""))
        # name the casualties
        for m, el in _items:
            _b = sum(1 for i in range(len(_R[_B])) if (m, el) not in _prim(_B, i))
            _a2 = sum(1 for i in range(len(_R[_A])) if (m, el) not in _prim(_A, i))
            if _b/len(_R[_B]) > _a2/len(_R[_A]):
                _asec = any((m, el) in _sec(_B, i) for i in range(len(_R[_B])))
                print(f"        LOST {m}:{el}" + ("  (demoted to secondary)" if _asec else ""))

    # ---- 3. the registered decision ---------------------------------------------
    print()
    EV.ablate(_R, _REF, closed=_CL, title="SEC vs v2")
    for _k in ("precision", "recall", "f1"):
        EV.paired(_R, _A, _B, _REF, key=_k)
    for _c in ("port", "signal", "signal-field"):
        EV.paired(_R, _A, _B, _REF, cls=_c, closed=_CL)
    print("\nDECISION RULE (registered before the run, ABLATION_LOG_V2.md section 7):")
    print("  primary list did NOT shrink, or <50% of departures moved -> REJECT (addition, not")
    print("     reassignment) -- this decides the arm whatever precision did")
    print("  precision up AND recall drop <=0.03 AND hard checks pass  -> promote at n=5")
    print("  recall drop 0.03-0.06 -> indeterminate, re-run at n=5 (EXPECTED: the arm sits on")
    print("     the guard by design, oracle recall cost is -0.036)")
    print("  any hard check fails  -> reject regardless of precision")
else:
    print(f"skipped: this cell is for the v2sec arm, VERSION is {VERSION}")


In [ ]:
# RUN-ALL GUARD. Reads the v2s3 runs.
if VERSION == "v2s3":
    # ===== S-3 DECISION CELL -- run AFTER v2s3 finishes =====
    # Three things are checked here, in this order, because S-3 can fail in three ways and the
    # aggregate hides two of them.
    importlib.reload(EV)
    import json as _json, statistics as _st
    from collections import Counter as _Counter

    _A, _B = "v2", "v2s3"
    _R = EV.collect([_A, _B])
    _REF = EV.load_refs()["gt"]
    _CL  = EV.load_closed()
    _M   = EV.common_modules(_R, _REF)

    # ---- 1. IS IT A CRITERION, OR DID IT BECOME A FAN-OUT DIAL? -------------------
    # X-3 intended a selective criterion and delivered a volume cut: elements/concept
    # 4.47 -> 4.15 with precision null. If S-3 moves elements/concept more than it moves
    # precision, it is turning the same class-blind dial and the precision gain is not the
    # mechanism claimed -- whatever the aggregate says.
    print("1. MECHANISM -- criterion or volume dial?")
    for _v in (_A, _B):
        _per = []
        for _i in range(len(_R[_v])):
            _ne = _nc = 0
            for _m in sorted(_M):
                _d = _json.load(open(f"assets_tuning18_{_v}_r{_i}/{_m}.json", encoding="utf-8"))
                _nc += len(_d.get("ConceptualAssets", [])); _ne += len(_d.get("Assets", []))
            _per.append(_ne / _nc if _nc else 0)
        print(f"     {_v:6s} elements/concept {_st.mean(_per):.2f}   concepts/module "
              f"{_nc/len(_M):.2f}")
    print("     v2 reference: 4.47 el/concept. R-8 moved it to 3.76, X-3 to 4.15.")
    print("     S-3 SHOULD LEAVE IT NEAR 4.47 -- it removes a class, not a quantity.\n")

    # ---- 2. HARD CHECKS -- both halves of the rule, independent of the aggregate --
    # The arm is rejected if these fail whatever precision did.
    #   group A: traffic that LEAVES THE DESIGN -- exception (2) must protect these.
    #            These are exactly the elements X-3R-8 destroyed (port recall -0.120).
    #   group B: the DECISION signals the rule redirects to, inside the interconnect
    #            module itself -- the rule must not suppress the module's real assets.
    _HARD = {"A_leaves_design": [("neorv32_uart", "uart_rxd_i"), ("neorv32_uart", "uart_txd_o"),
                                 ("neorv32_spi", "spi_dat_i"),   ("neorv32_twi", "twi_sda_i")],
             "B_decisions":     [("neorv32_bus", "sel"), ("neorv32_bus", "stb"),
                                 ("neorv32_bus", "port_sel"), ("neorv32_bus", "state")],
             # group C: the CONTROL record the module READS AND ACTS ON. The rule's last clause
             # exempts these explicitly. Measured: applying the fabric test to the control record
             # instead would cost 16.7 TP/repeat and drop recall 0.925 -> 0.775. These 17 elements
             # ARE that population, so they are the direct test of whether the model respected the
             # boundary or generalised the rule to every record it saw.
             "C_acted_on":      [("neorv32_wdt", "ctrl.enable"), ("neorv32_wdt", "ctrl.lock"),
                                 ("neorv32_wdt", "ctrl.timeout"), ("neorv32_uart", "ctrl.baud"),
                                 ("neorv32_uart", "ctrl.enable"), ("neorv32_spi", "ctrl.prsc"),
                                 ("neorv32_twi", "ctrl.cdiv"),
                                 ("neorv32_cpu_cp_muldiv", "ctrl.state")]}
    print("2. HARD CHECKS (reject the arm if any group-A or group-B element is lost)")
    for _g, _items in _HARD.items():
        print(f"   {_g}")
        for _mod, _el in _items:
            _row = []
            for _v in (_A, _B):
                _hit = sum(1 for _r in _R[_v]
                           if any(_x[1] == _el for _x in EV._as_run(_r).get(_mod, [])))
                _row.append(f"{_v} {_hit}/{len(_R[_v])}")
            _flag = ""
            _b = sum(1 for _r in _R[_B] if any(_x[1] == _el for _x in EV._as_run(_r).get(_mod, [])))
            _a = sum(1 for _r in _R[_A] if any(_x[1] == _el for _x in EV._as_run(_r).get(_mod, [])))
            if _b < _a: _flag = "   <== LOST, arm rejected"
            print(f"     {_mod}:{_el:14s} " + "  ".join(_row) + _flag)

    # ---- 3. DID IT REMOVE THE INTENDED POPULATION? -------------------------------
    # The headroom was 76.0 FP/repeat whose DECLARED TYPE is the shared interconnect record.
    # If precision rose but this number did not fall, the gain came from somewhere else.
    import pathlib as _pl
    _TY = {}
    _tm = {}
    for _f in sorted(_pl.Path("parsed_tuning18").glob("*.json")):
        _d = _json.loads(_f.read_text(encoding="utf-8"))
        for _e in _d.get("ports", []) + _d.get("signals", []):
            _t = (_e.get("type") or "").strip()
            _TY[(_f.stem, _e["name"])] = _t
            _tm.setdefault(_t, set()).add(_f.stem)
    _FAB = {_t for _t, _ms in _tm.items()
            if len(_ms) >= 6 and not _t.lower().startswith("std_") and _t.lower() != "ctrl_t"}
    print(f"\n3. TARGET POPULATION -- shared fabric record types {sorted(_FAB)}")
    for _v in (_A, _B):
        _scs = [EV.score(_r, _REF, True, only=_M) for _r in _R[_v]]
        _fp = sum(1 for _s in _scs for _m in _M for _f2 in _s["per_module"][_m]["fp"]
                  if _TY.get((_m, _f2.split(".")[0]), "") in _FAB) / len(_scs)
        _tot = _st.mean(_s["fp"] for _s in _scs)
        print(f"     {_v:6s} fabric-typed FP {_fp:6.1f}/rep   of {_tot:6.1f} total FP "
              f"({100*_fp/_tot:4.1f}%)")
    print("     v2 baseline is 76.0/rep. A real S-3 effect drives this toward zero.\n")

    # ---- 4. the registered decision ---------------------------------------------
    EV.ablate(_R, _REF, closed=_CL, title="S-3 vs v2")
    for _k in ("precision", "recall", "f1"):
        EV.paired(_R, _A, _B, _REF, key=_k)
    for _c in ("port", "signal", "signal-field"):
        EV.paired(_R, _A, _B, _REF, cls=_c, closed=_CL)
    print("\nDECISION RULE (registered before the run, ABLATION_LOG_V2.md section 7):")
    print("  precision up AND recall drop <= 0.03 AND all hard checks pass -> promote at n=5")
    print("  any hard check fails                                          -> reject")
    print("  precision null                                                -> record, do NOT re-word")
else:
    print(f"skipped: this cell is for the v2s3 arm, VERSION is {VERSION}")


In [19]:
# ===== What this version missed and what it invented =====
if res:
    EV.table(res, f"{VERSION} vs manual GT", show_names=True)


v2rp vs manual GT
                        ref  emit   TP   FP   FN      P      R     F1    obj
----------------------------------------------------------------------------
bus                      11   126   11  115    0  0.087  1.000  0.161  1.000
cache                     6    43    3   40    3  0.070  0.500  0.122  0.333
cpu                      14    45   12   33    2  0.267  0.857  0.407  0.917
cpu_cp_cfu               11    23    9   14    2  0.391  0.818  0.529  0.556
cpu_cp_muldiv            12    25   10   15    2  0.400  0.833  0.541  0.700
cpu_pmp                   6    18    6   12    0  0.333  1.000  0.500  0.667
debug_dtm                 5    24    3   21    2  0.125  0.600  0.207  1.000
hwspinlock                2     7    2    5    0  0.286  1.000  0.444  1.000
imem                      3    11    3    8    0  0.273  1.000  0.429  0.333
spi                       8    42    8   34    0  0.190  1.000  0.320  0.625
sys                       2    15    2   13    0  0.133  1

In [ ]:
# RUN-ALL GUARD. Issues judge API calls, so it is OPT-IN. Set RUN_T1 = True and
# DIAG_VERSION to the arm you want analysed. Left False so a Run All sweep never
# spends on a diagnostic nobody asked for.
RUN_T1 = False
if RUN_T1:
    # ===== STAGE C SUBSTAGE -- T-1: conception failure vs binding failure =====
    # DIAGNOSTIC ONLY. Classifies MISSES; it is not a metric and no arm is accepted or rejected
    # on it. v01c4 changes the output contract and asks for reasoning that was internal, either
    # of which can move behaviour on its own -- so its recall delta is CONFOUNDED, and asking
    # for the concept list BEFORE the assets may itself cost recall by forcing early commitment.
    import importlib, diagnose_step1 as DG
    importlib.reload(DG)
    from collections import Counter

    # Pointing this at the wrong version is not a harmless mistake: the first run of this cell
    # analysed v01c4 -- the one version whose concepts generate_assets() had stripped -- and
    # reported 100% "conception failure" while v01c5 sat on disk fully populated. The guard
    # below refuses to analyse a run that has no concept list anywhere, because the only
    # output such a run can produce is a confident wrong answer.
    # DIAG_VERSION drives judge API calls. Left explicit rather than following
    # VERSION, because this diagnostic is opt-in: set it to the arm you want
    # analysed and set RUN_T1 to True. Under Run All it stays off, so a sweep
    # never spends on a diagnostic nobody asked for.
    DIAG_VERSION = "v2"
    _dirs = sorted(Path(".").glob(f"assets_tuning18_{DIAG_VERSION}_r*"))
    if _dirs and not any(v["concepts"] for v in DG.load_concept_runs(_dirs[0]).values()):
        raise RuntimeError(
            f"{DIAG_VERSION} r0 has no ConceptualAssets in any module. Either the prompt did "
            f"not ask for them, or generate_assets() stripped them. Analysing this would "
            f"report 100% conception failure, and that would be an artefact.")

    if not _dirs:
        print(f"no {DIAG_VERSION} runs yet -- set VERSION = '{DIAG_VERSION}' and run Stage B")
    else:
        _runs = EV.collect([DIAG_VERSION])
        _mods = EV.common_modules(_runs, REFS["gt"])

        # The judge. Blind by construction: it never learns whether an element was found or
        # missed, nor which concept the model itself linked it to.
        def _call(system, user):
            # effort='low': the judge answers with one token (an id or NONE), and reasoning
            # shares the output budget. 2048 is a ceiling, not a reservation.
            return client.complete_text(system, user, max_tokens=2048, effort="low")
        JUDGE = DG.make_judge(_call)

        # ---- CALIBRATION FIRST. Not optional: with a judge that always picks a concept the
        # split reads 100% (b), with one that always says NONE it reads 100% (a). The `any`
        # rate bounds every (a) verdict, because a judge that answers NONE on links the model
        # explicitly made will answer NONE on misses too.
        _cal = DG.calibrate_judge(DG.load_concept_runs(_dirs[0]), REFS["gt"],
                                  mods=_mods, judge=JUDGE, limit=40)
        print(f"JUDGE CALIBRATION on {_cal['n']} FOUND references (model asserted the link):")
        print(f"  exact concept id agreement : {_cal['exact']:.2f}")
        print(f"  any-concept agreement      : {_cal['any']:.2f}   <- bounds the (a) verdicts")
        if _cal["any"] < 0.80:
            print(f"  [warn] judge answers NONE on {100*(1-_cal['any']):.0f}% of links the model "
                  f"MADE. Every (a) below inherits that false-negative rate.")

        # ---- PRIMARY SPLIT, over DISTINCT references per repeat.
        agg, K = Counter(), len(_dirs)
        for _d in _dirs:
            _rows, _t = DG.split_misses(DG.load_concept_runs(_d), REFS["gt"],
                                        mods=_mods, judge=JUDGE)
            agg.update(_t)
        n = sum(agg.values())
        print(f"\n(a)/(b) SPLIT -- {n/K:.1f} distinct misses per repeat, {K} repeats")
        for k in DG.VERDICTS:
            if agg[k]:
                print(f"  {agg[k]/K:6.1f}/repeat ({100*agg[k]/n:5.1f}%)  {k}")
        print("  (b) is sound; (a) is weaker -- absence from the emitted list is not proof the")
        print("  model never conceived it. Do not present them as symmetric.")

        # ---- CROSS-CHECK. Disagreements are the rows worth reading by hand.
        _cr, _cor = DG.corroborating_evidence(EV.load_run(_dirs[0]), REFS["gt"], _mods)
        print("\ncorroboration on r0 (sibling / secondary, NOT verdicts):")
        for k in DG.TIERS:
            if _cor[k]:
                print(f"  {_cor[k]:4d}  {k}")

        # ---- Q5: are FPs over-binding of legitimate concepts, or bad concepts?
        _out, _per = DG.concept_precision(DG.load_concept_runs(_dirs[0]), REFS["gt"], _mods)
        print("\nconcept-level precision (r0):")
        for k, v in _out.most_common():
            print(f"  {v:4d}  {k}")

        # ---- UNBOUND CONCEPTS: a third outcome, distinct from (a) and (b).
        # The model conceived an idea and bound no element to it. The prompt explicitly invites
        # this ("a conceptual asset with no closed-set element still belongs here"), so it is
        # not a contract violation -- it is the model reporting that the closed set cannot
        # express something it considered. cache.inval_i is the known case: class `absent`,
        # genuinely not in the parsed set because of the C-03 revision skew.
        _ub, _rows = DG.concept_binding(DG.load_concept_runs(_dirs[0]), _mods)
        print("\nconcept binding (r0):")
        for k, v in _ub.most_common():
            print(f"  {v:4d}  {k}")
        if _rows:
            print("  unbound concepts, first 8:")
            for r in _rows[:8]:
                print(f"    {r['module'].replace('neorv32_',''):16s} {r['id']:4s} "
                      f"[{r['objective'] or '?'}] {r['concept'][:58]}")
else:
    print(f"skipped: this cell is for the T-1 diagnostic (RUN_T1), VERSION is {VERSION}")
